# 20 · BirdNET vs. ResNet152V2 — Inferencia de BirdNET sobre el test set

Este notebook aplica **BirdNET V2.4** al conjunto de test del proyecto para compararlo
con el modelo propio (**ResNet152V2**, 667 especies). Como ya no se conservan los audios
originales —solo los espectrogramas—, se **re-descargan las grabaciones originales de
Xeno-canto** usando el ID que quedó codificado en el nombre de cada imagen
(`<xc_id>_<offset>.jpeg`, ver `src/spectograms.py`).

Decisiones de diseño (acordadas):
- **Comparación por grabación (top-1)**: BirdNET segmenta en ventanas de 3 s; se agregan a
  una predicción por grabación.
- **Espacio de clases restringido** a las especies del dataset que BirdNET conoce
  (`custom_species_list`) → comparación justa.
- **Muestra estratificada** (N grabaciones por especie) para una primera corrida manejable.

> **Kernel**: ejecuta este notebook con el entorno aislado **`.venv-birdnet`** (Python 3.11 +
> `birdnet`), NO con el `.venv` del proyecto (que tiene TensorFlow 2.19 para el ResNet).

Salidas (en `src/data/`): `birdnet_species_coverage.csv`, `birdnet_sample_recordings.csv`,
`birdnet_segments.csv`, `birdnet_predictions.csv`.

## Requisitos

**1. Entorno aislado con BirdNET** (ya creado si seguiste el plan):
```bash
uv venv .venv-birdnet --python 3.11
uv pip install --python .venv-birdnet/bin/python --index-url https://pypi.org/simple birdnet requests scikit-learn ipykernel
.venv-birdnet/bin/python -m ipykernel install --user --name birdnet --display-name "Python (birdnet)"
```
Selecciona el kernel **Python (birdnet)** para este notebook.

**2. Sin API key**: los audios se descargan por la URL pública de Xeno-canto
(`https://xeno-canto.org/<id>/download`), que **no requiere clave** (el mismo acceso
que usaba `xenopy`). Solo hace falta conexión a internet.

In [1]:
import os, sys, subprocess

import numpy as np
import pandas as pd

# Nota: la inferencia de BirdNET se ejecuta en un SUBPROCESO (ver celda 5), no en este
# kernel. BirdNET usa multiprocessing y correrlo aquí con 'fork' causa deadlocks en
# macOS/Jupyter; el subproceso lo ejecuta con 'spawn', que es estable.

sys.path.append("../src")
import birdnet_utils as bu

# ---------------- Configuración ----------------
ROOT        = os.path.abspath("..")
IMAGES_ROOT = os.path.join(ROOT, "src/data/images_test/images_spectograms")
DATA_DIR    = os.path.join(ROOT, "src/data")
AUDIO_DIR   = os.path.join(ROOT, "data/audio_test_sample")

# N_PER_SPECIES: grabaciones por especie a evaluar.
#   entero (p. ej. 2) -> muestra rápida  |  None -> TODO el test (~18k audios:
#   la 1ª corrida tarda por descarga+inferencia, pero es reanudable por checkpointing).
N_PER_SPECIES = 20
HOW_AGG       = "max"  # agregación de segmentos -> grabación: 'max' o 'mean'
TOP_K         = 1      # nº de especies por segmento que devuelve BirdNET
SEED          = 42

# Los audios se descargan por la URL pública de Xeno-canto (sin API key).
os.makedirs(AUDIO_DIR, exist_ok=True)
print("images :", IMAGES_ROOT)
print("audio  :", AUDIO_DIR)

images : /Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms
audio  : /Users/camcortes/Documents/birds-sounds/data/audio_test_sample


In [2]:
# 1) Índice del test set a nivel de grabación (a partir de los espectrogramas)
index = bu.build_test_index(IMAGES_ROOT)
print("grabaciones:", index.recording_id.nunique(),
      "| especies:", index.species.nunique(),
      "| chunks:", int(index.n_chunks.sum()))
index.head()

grabaciones: 18227 | especies: 667 | chunks: 32279


,recording_id,species,n_chunks,chunk_paths
0,104508,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...
1,119693,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...
2,119694,Acropternis orthonyx,2,[/Users/camcortes/Documents/birds-sounds/src/d...
3,121142,Acropternis orthonyx,3,[/Users/camcortes/Documents/birds-sounds/src/d...
4,127718,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...


In [3]:
# 2) Cargar BirdNET V2.4 y calcular la cobertura de especies del dataset
import birdnet

model = birdnet.load("acoustic", "2.4", "tf")
print("BirdNET V2.4 |", model.n_species, "especies |",
      model.get_sample_rate(), "Hz |", model.get_segment_size_s(), "s/segmento")

# Persistir la lista de especies del modelo
species_list = list(model.species_list)
with open(os.path.join(DATA_DIR, "birdnet_v2.4_species_list.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(species_list))
labels_df = bu.birdnet_labels_to_df(species_list)

# Cobertura: cruzar las especies del dataset con la taxonomía de BirdNET
dataset_species = sorted(index.species.unique())
coverage = bu.match_species(dataset_species, labels_df)
gf = bu.get_genus_family_map()
coverage["family"] = coverage["species"].map(lambda s: bu.family_of(s, gf))
coverage.to_csv(os.path.join(DATA_DIR, "birdnet_species_coverage.csv"), index=False)

n_cov = int(coverage.covered.sum())
print(f"Cobertura BirdNET: {n_cov}/{len(coverage)} ({100*n_cov/len(coverage):.1f}%)")
print("NO cubiertas:", coverage.loc[~coverage.covered, "species"].tolist())

BirdNET V2.4 | 6522 especies | 48000 Hz | 3.0 s/segmento
Cobertura BirdNET: 664/667 (99.6%)
NO cubiertas: ['Haplospiza rustica', 'Hylopezus fulviventris', 'Myiophobus roraimae']


In [4]:
# 3) Muestra estratificada (solo especies que BirdNET puede predecir)
covered_species = set(coverage.loc[coverage.covered, "species"])
sample = bu.stratified_sample(index, N_PER_SPECIES,
                              covered_species=covered_species, seed=SEED)
sci2label = dict(zip(coverage.species, coverage.birdnet_label))
sample["birdnet_label_true"] = sample.species.map(sci2label)
print("grabaciones en la muestra:", len(sample), "| especies:", sample.species.nunique())
sample.head()

grabaciones en la muestra: 11702 | especies: 664


,recording_id,species,n_chunks,chunk_paths,birdnet_label_true
0,621780,Acropternis orthonyx,2,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
1,374408,Acropternis orthonyx,4,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
2,374409,Acropternis orthonyx,3,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
3,782306,Acropternis orthonyx,3,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
4,51066,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo


In [5]:
# 4) Descargar los audios originales de Xeno-canto (checkpointing: omite los ya bajados)
import requests
session = requests.Session()

paths, ok, fail = [], 0, 0
for i, row in enumerate(sample.itertuples(), 1):
    p = bu.download_xc_recording(row.recording_id, AUDIO_DIR,
                                 species=row.species, session=session)
    paths.append(p)
    ok += bool(p); fail += (not p)
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}  ok={ok}  fallidos={fail}")

sample["audio_path"] = paths
sample.to_csv(os.path.join(DATA_DIR, "birdnet_sample_recordings.csv"), index=False)
print(f"Descarga terminada: OK={ok}  fallidos/eliminados={fail}  total={len(sample)}")

  50/11702  ok=50  fallidos=0
  100/11702  ok=100  fallidos=0
  150/11702  ok=150  fallidos=0
  200/11702  ok=200  fallidos=0
  250/11702  ok=250  fallidos=0
  300/11702  ok=300  fallidos=0
  350/11702  ok=350  fallidos=0
  400/11702  ok=400  fallidos=0
  450/11702  ok=450  fallidos=0
  500/11702  ok=500  fallidos=0
  550/11702  ok=550  fallidos=0
  600/11702  ok=600  fallidos=0
  650/11702  ok=650  fallidos=0
  700/11702  ok=700  fallidos=0
  750/11702  ok=750  fallidos=0
  800/11702  ok=800  fallidos=0
  850/11702  ok=850  fallidos=0
  900/11702  ok=899  fallidos=1
  950/11702  ok=949  fallidos=1
  1000/11702  ok=999  fallidos=1
  1050/11702  ok=1049  fallidos=1
  1100/11702  ok=1099  fallidos=1
  1150/11702  ok=1149  fallidos=1
  1200/11702  ok=1199  fallidos=1
  1250/11702  ok=1249  fallidos=1
  1300/11702  ok=1299  fallidos=1
  1350/11702  ok=1349  fallidos=1
  1400/11702  ok=1399  fallidos=1
  1450/11702  ok=1449  fallidos=1
  1500/11702  ok=1499  fallidos=1
  1550/11702  ok=1549

In [6]:
# 5) Inferencia BirdNET en un SUBPROCESO (spawn) — evita el deadlock de multiprocessing
#    (fork) en Jupyter y es rápido: pre-decodifica a 48 kHz y usa predict_arrays en paralelo.
#    El subproceso descarta internamente los audios ilegibles/corruptos.
custom_species = sorted(coverage.loc[coverage.covered, "birdnet_label"].dropna().unique())

infer_in    = os.path.join(DATA_DIR, "_birdnet_infer_input.csv")
custom_file = os.path.join(DATA_DIR, "_birdnet_custom_species.txt")
seg_path    = os.path.join(DATA_DIR, "birdnet_segments.csv")
sample[["recording_id", "species", "audio_path"]].to_csv(infer_in, index=False)
with open(custom_file, "w", encoding="utf-8") as f:
    f.write("\n".join(custom_species))

cmd = [sys.executable, "../src/run_birdnet_inference.py",
       "--input-csv", infer_in, "--custom-species-file", custom_file,
       "--out-csv", seg_path, "--n-producers", "6", "--top-k", str(TOP_K)]
print("Ejecutando BirdNET en subproceso (spawn)…  (esto puede tardar varios minutos)")
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-3000:])
    raise RuntimeError("La inferencia BirdNET (subproceso) falló; revisa el STDERR de arriba.")

seg_df = pd.read_csv(seg_path)
print("filas (segmento x especie):", len(seg_df),
      "| grabaciones con predicción:", seg_df.recording_id.nunique())
seg_df.head()

Ejecutando BirdNET en subproceso (spawn)…  (esto puede tardar varios minutos)
 audios/s | 9.1 min
[infer] 4000/11699 audios | ~3999 grabaciones | 7.0 audios/s | 9.5 min
[infer] 4200/11699 audios | ~4199 grabaciones | 7.0 audios/s | 10.0 min
[infer] 4400/11699 audios | ~4398 grabaciones | 7.0 audios/s | 10.4 min
[infer] 4600/11699 audios | ~4598 grabaciones | 7.1 audios/s | 10.8 min
[infer] 4800/11699 audios | ~4798 grabaciones | 7.1 audios/s | 11.3 min
[infer] 5000/11699 audios | ~4998 grabaciones | 7.1 audios/s | 11.8 min
[infer] 5200/11699 audios | ~5198 grabaciones | 7.1 audios/s | 12.2 min
[infer] 5400/11699 audios | ~5398 grabaciones | 7.1 audios/s | 12.6 min
[infer] 5600/11699 audios | ~5598 grabaciones | 7.2 audios/s | 13.0 min
[infer] 5800/11699 audios | ~5798 grabaciones | 7.2 audios/s | 13.4 min
[infer] 6000/11699 audios | ~5998 grabaciones | 7.2 audios/s | 13.9 min
[infer] 6200/11699 audios | ~6198 grabaciones | 7.2 audios/s | 14.4 min
[infer] 6400/11699 audios | ~6398 graba

,recording_id,species_true,start_time,end_time,species_name,confidence
0,621780,Acropternis orthonyx,0.0,3.0,Acropternis orthonyx_Ocellated Tapaculo,0.999824
1,621780,Acropternis orthonyx,3.0,6.0,Acropternis orthonyx_Ocellated Tapaculo,0.970204
2,621780,Acropternis orthonyx,6.0,9.0,Acropternis orthonyx_Ocellated Tapaculo,0.722943
3,621780,Acropternis orthonyx,9.0,12.0,Acropternis orthonyx_Ocellated Tapaculo,0.996603
4,621780,Acropternis orthonyx,12.0,15.0,Acropternis orthonyx_Ocellated Tapaculo,0.997468


In [7]:
# seg_df = seg_df.sort_values(by='confidence', ascending=True)
# seg_df.drop_duplicates(subset='recording_id', keep='first')
# seg_df.head()

In [8]:
# 6) Agregar a una predicción top-1 por grabación
rows = []
for rec, g in seg_df.groupby("recording_id"):
    top_label, conf = bu.birdnet_top1_from_dataframe(g, how=HOW_AGG)
    rows.append({
        "recording_id": rec,
        "species_true": g.species_true.iloc[0],
        "birdnet_label": top_label,
        "species_pred": bu.birdnet_label_to_scientific(top_label),
        "confidence": conf,
    })
pred = pd.DataFrame(rows)
pred.to_csv(os.path.join(DATA_DIR, "birdnet_predictions.csv"), index=False)
print("predicciones recording-level:", len(pred))
pred.head()

predicciones recording-level: 11695


,recording_id,species_true,birdnet_label,species_pred,confidence
0,13,Pipreola arcuata,Pipreola arcuata_Barred Fruiteater,Pipreola arcuata,0.115836
1,90,Platyrinchus coronatus,Platyrinchus coronatus_Golden-crowned Spadebill,Platyrinchus coronatus,0.995942
2,169,Asthenes fuliginosa,Asthenes fuliginosa_White-chinned Thistletail,Asthenes fuliginosa,0.957592
3,435,Pionus menstruus,Pionus menstruus_Blue-headed Parrot,Pionus menstruus,0.964426
4,465,Ammodramus humeralis,Ammodramus humeralis_Grassland Sparrow,Ammodramus humeralis,0.998865


In [9]:
# 7) Métricas de BirdNET sobre la muestra (recording-level)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

valid = pred.dropna(subset=["species_pred"])
acc = accuracy_score(valid.species_true, valid.species_pred)
f1m = f1_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)
prm = precision_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)
rcm = recall_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)

n_audios = int(sample.audio_path.notna().sum())
print(f"BirdNET V2.4 · recording-level · N_PER_SPECIES={N_PER_SPECIES}")
print(f"  grabaciones en la muestra   : {len(sample)}")
print(f"  descargas fallidas          : {fail}")
print(f"  audios omitidos (corruptos) : {n_audios - len(pred)}")
print(f"  grabaciones evaluadas       : {len(valid)}")
print(f"  accuracy                    : {acc:.4f}")
print(f"  F1 macro                    : {f1m:.4f}")
print(f"  precision macro             : {prm:.4f}")
print(f"  recall macro                : {rcm:.4f}")
print("\nSiguiente paso: ejecuta 21_resnet_recording_eval_and_compare.ipynb (kernel .venv del proyecto).")

BirdNET V2.4 · recording-level · N_PER_SPECIES=20
  grabaciones en la muestra   : 11702
  descargas fallidas          : 3
  audios omitidos (corruptos) : 4
  grabaciones evaluadas       : 11695
  accuracy                    : 0.9027
  F1 macro                    : 0.8995
  precision macro             : 0.9122
  recall macro                : 0.8992

Siguiente paso: ejecuta 21_resnet_recording_eval_and_compare.ipynb (kernel .venv del proyecto).
